In [1]:
import pandas as pd
import numpy as np
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from lifelines import CoxPHFitter
from sklearn.svm import NuSVR
from sklearn.preprocessing import normalize
from scipy.optimize import nnls

base = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'
print("All imports successful ✅")

All imports successful ✅


In [2]:
# Load all streams
expr     = pd.read_csv(f'{base}/data/processed/expression_matrix.csv', index_col=0)
dysreg   = pd.read_csv(f'{base}/data/processed/dysregulation_scores.csv', index_col=0)
immune   = pd.read_csv(f'{base}/data/processed/immune_features_cibersort.csv', index_col=0)
clinical = pd.read_csv(f'{base}/data/processed/clinical_survival.csv', index_col=0)

# Align patients
common   = expr.index.intersection(dysreg.index).intersection(
           immune.index).intersection(clinical.index)
expr     = expr.loc[common]
dysreg   = dysreg.loc[common]
immune   = immune.loc[common]
clinical = clinical.loc[common]

# Clinical features
age           = clinical[['age']].copy()
gender        = (clinical['gender'] == 'male').astype(float).to_frame()
stage_dummies = pd.get_dummies(clinical['stage_group'], prefix='stage')
stage_dummies = stage_dummies.drop(columns=['stage_Stage I'], errors='ignore')
clinical_features = pd.concat([age, gender, stage_dummies], 
                               axis=1).astype(float).fillna(0)

# Survival labels
y = np.array(
    [(bool(e), t) for e, t in zip(clinical['event'], clinical['survival_time'])],
    dtype=[('event', bool), ('time', float)])

# ── RANK TRANSFORMATION ───────────────────────────────────────────
# Convert to percentile ranks within each patient
# rank(axis=1) = rank across genes for each patient
# pct=True converts to 0-1 range
# This makes features platform-invariant
expr_ranked  = expr.rank(axis=1, pct=True)
dysreg_ranked = dysreg.rank(axis=1, pct=True)

print(f"Patients:       {len(common)}")
print(f"Expression:     {expr_ranked.shape}")
print(f"Dysregulation:  {dysreg_ranked.shape}")
print(f"\nOriginal expression range: {expr.values.min():.2f} to {expr.values.max():.2f}")
print(f"Ranked expression range:   {expr_ranked.values.min():.4f} to {expr_ranked.values.max():.4f}")
print(f"\nOriginal dysreg range: {dysreg.values.min():.2f} to {dysreg.values.max():.2f}")
print(f"Ranked dysreg range:   {dysreg_ranked.values.min():.4f} to {dysreg_ranked.values.max():.4f}")

Patients:       478
Expression:     (478, 1000)
Dysregulation:  (478, 819)

Original expression range: 0.00 to 20.45
Ranked expression range:   0.0040 to 1.0000

Original dysreg range: -2.72 to 80.09
Ranked dysreg range:   0.0012 to 1.0000


In [3]:
# Load Lasso genes
cox_lasso   = pickle.load(open(f'{base}/models/cox_lasso_expression.pkl', 'rb'))
gene_list   = json.load(open(f'{base}/models/gene_list.json'))
coefs       = cox_lasso.coef_[:, 0]
lasso_genes = [g for g, s in zip(gene_list, coefs != 0) if s]

# Lasso-selected expression ranked
expr_lasso_ranked = expr_ranked[lasso_genes].copy()
expr_lasso_ranked.columns = [f"{g}_expr" for g in lasso_genes]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_cindex = []

print("Running leakage-free 5-fold CV — RANKED features")
print(f"{'Fold':<6} {'Test C-index':<12}")
print("-" * 20)

for fold, (train_idx, test_idx) in enumerate(kf.split(expr_lasso_ranked, y['event']), 1):

    expr_train,     expr_test     = expr_lasso_ranked.iloc[train_idx], expr_lasso_ranked.iloc[test_idx]
    dysreg_train,   dysreg_test   = dysreg_ranked.iloc[train_idx],     dysreg_ranked.iloc[test_idx]
    immune_train,   immune_test   = immune.iloc[train_idx],            immune.iloc[test_idx]
    clinical_train, clinical_test = clinical_features.iloc[train_idx], clinical_features.iloc[test_idx]
    y_train,        y_test        = y[train_idx],                      y[test_idx]

    times_train  = y_train['time'].copy()
    events_train = y_train['event'].copy()
    times_test   = y_test['time'].copy()
    events_test  = y_test['event'].copy()

    # Cox dysregulation selection inside fold on RANKED features
    cox_pvals_d = {}
    for gene in dysreg_train.columns:
        try:
            df_tmp = pd.DataFrame({'T': times_train, 'E': events_train,
                                   'gene': dysreg_train[gene].values})
            cph = CoxPHFitter()
            cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
            cox_pvals_d[gene] = cph.summary['p'].values[0]
        except:
            cox_pvals_d[gene] = 1.0
    top_dysreg = pd.Series(cox_pvals_d).nsmallest(20).index

    dysreg_tr = dysreg_train[top_dysreg].copy()
    dysreg_te = dysreg_test[top_dysreg].copy()
    dysreg_tr.columns = [f"{g}_dysreg" for g in top_dysreg]
    dysreg_te.columns = [f"{g}_dysreg" for g in top_dysreg]

    # Interaction features
    inter_train = pd.DataFrame({
        'stageIII_x_M2':   (clinical_train['stage_Stage III'] *
                            immune_train['Macrophages M2']).values,
        'stageIV_x_CD8':   (clinical_train['stage_Stage IV'] *
                            immune_train['T cells CD8']).values,
        'age_x_stageIII':  (clinical_train['age'] *
                            clinical_train['stage_Stage III']).values,
        'stageIII_x_Treg': (clinical_train['stage_Stage III'] *
                            immune_train['T cells regulatory (Tregs)']).values,
        'M2_x_CD8':        (immune_train['Macrophages M2'] *
                            immune_train['T cells CD8']).values,
    }, index=clinical_train.index)

    inter_test = pd.DataFrame({
        'stageIII_x_M2':   (clinical_test['stage_Stage III'] *
                            immune_test['Macrophages M2']).values,
        'stageIV_x_CD8':   (clinical_test['stage_Stage IV'] *
                            immune_test['T cells CD8']).values,
        'age_x_stageIII':  (clinical_test['age'] *
                            clinical_test['stage_Stage III']).values,
        'stageIII_x_Treg': (clinical_test['stage_Stage III'] *
                            immune_test['T cells regulatory (Tregs)']).values,
        'M2_x_CD8':        (immune_test['Macrophages M2'] *
                            immune_test['T cells CD8']).values,
    }, index=clinical_test.index)

    X_train = pd.concat([expr_train, dysreg_tr, immune_train,
                          clinical_train, inter_train], axis=1)
    X_test  = pd.concat([expr_test,  dysreg_te, immune_test,
                          clinical_test,  inter_test], axis=1)

    scaler    = StandardScaler()
    X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test_s  = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

    model = GradientBoostingSurvivalAnalysis(
        n_estimators=300, learning_rate=0.05, max_depth=2,
        min_samples_split=20, min_samples_leaf=10,
        subsample=0.8, random_state=42)
    model.fit(X_train_s, y_train)

    ci = concordance_index_censored(
        events_test.astype(bool), times_test,
        model.predict(X_test_s))[0]

    fold_cindex.append(ci)
    print(f"{fold:<6} {ci:.4f}")

print("-" * 20)
print(f"\nRanked features C-index: {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")
print(f"\nComparison:")
print(f"  Original features (leakage-free): 0.702 ± 0.057")
print(f"  Ranked features:                  {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")

Running leakage-free 5-fold CV — RANKED features
Fold   Test C-index
--------------------
1      0.6398
2      0.7862
3      0.6060
4      0.7031
5      0.8020
--------------------

Ranked features C-index: 0.707 ± 0.078

Comparison:
  Original features (leakage-free): 0.702 ± 0.057
  Ranked features:                  0.707 ± 0.078


In [4]:
# Load GSE68465 data we already processed
import GEOparse

# Reload expression
gse = GEOparse.get_GEO(geo="GSE68465",
                        destdir=f'{base}/data/external/',
                        silent=True)

gsm_data = {}
for gsm_name, gsm in gse.gsms.items():
    if gsm.table is not None and len(gsm.table) > 0:
        gsm_data[gsm_name] = gsm.table.set_index('ID_REF')['VALUE']

expr_raw_ext = pd.DataFrame(gsm_data).T

# Platform annotation
gpl = gse.gpls['GPL96']
probe_to_gene = gpl.table.set_index('ID')['Gene Symbol'].dropna()
probe_to_gene = probe_to_gene[probe_to_gene != '']

expr_filtered = expr_raw_ext[[c for c in expr_raw_ext.columns 
                               if c in probe_to_gene.index]]
expr_filtered.columns = [probe_to_gene[c] for c in expr_filtered.columns]
expr_filtered = expr_filtered.astype(float)
expr_filtered = expr_filtered.T.groupby(level=0).mean().T

# Log2 transform
expr_log2_ext = np.log2(expr_filtered + 1)

# ── RANK TRANSFORMATION on external data ─────────────────────────
expr_ranked_ext = expr_log2_ext.rank(axis=1, pct=True)

print(f"External expression ranked: {expr_ranked_ext.shape}")
print(f"Rank range: {expr_ranked_ext.values.min():.4f} to {expr_ranked_ext.values.max():.4f}")

# Reload clinical and survival from notebook 15
survival_records = []
for gsm_name, gsm in gse.gsms.items():
    record = {'sample_id': gsm_name}
    for c in gsm.metadata.get('characteristics_ch1', []):
        if ':' in c:
            key, val = c.split(':', 1)
            record[key.strip()] = val.strip()
    survival_records.append(record)

survival_df_ext = pd.DataFrame(survival_records).set_index('sample_id')

def parse_ptnm_stage(s):
    s = str(s).strip()
    try:
        n = int(s[2]) if 'N' in s and s[2].isdigit() else 0
        t = int(s[5]) if 'T' in s and s[5].isdigit() else 1
        if n == 0 and t in [1, 2]:
            return 'Stage I' if t == 1 else 'Stage II'
        elif n == 1 and t in [1, 2]:
            return 'Stage II'
        elif n == 2 or t in [3, 4]:
            return 'Stage III'
        else:
            return 'Stage I'
    except:
        return 'Unknown'

survival_df_ext['stage_group'] = survival_df_ext['disease_stage'].apply(parse_ptnm_stage)
survival_df_ext['survival_days'] = pd.to_numeric(
    survival_df_ext['months_to_last_contact_or_death'], errors='coerce') * 30.44
survival_df_ext = survival_df_ext[survival_df_ext['survival_days'].notna()]
survival_df_ext = survival_df_ext[survival_df_ext['vital_status'].isin(['Alive', 'Dead'])]

# Align
common_ext = expr_ranked_ext.index.intersection(survival_df_ext.index)
expr_ranked_ext  = expr_ranked_ext.loc[common_ext]
survival_df_ext  = survival_df_ext.loc[common_ext]

# Clinical features
age_ext    = pd.to_numeric(survival_df_ext['age'], errors='coerce').fillna(65)
gender_ext = (survival_df_ext['Sex'] == 'Male').astype(float)
stage_II   = (survival_df_ext['stage_group'] == 'Stage II').astype(float)
stage_III  = (survival_df_ext['stage_group'] == 'Stage III').astype(float)
stage_IV   = (survival_df_ext['stage_group'] == 'Stage IV').astype(float)

clinical_ext_df = pd.DataFrame({
    'age':             age_ext.values,
    'gender':          gender_ext.values,
    'stage_Stage II':  stage_II.values,
    'stage_Stage III': stage_III.values,
    'stage_Stage IV':  stage_IV.values
}, index=common_ext)

print(f"External patients: {len(common_ext)}")
print(f"Events: {(survival_df_ext['vital_status']=='Dead').sum()}")

External expression ranked: (462, 13515)
Rank range: 0.0001 to 1.0000
External patients: 442
Events: 236


In [5]:
from datetime import datetime

# Load GTEx reference for dysregulation
gtex_ref = json.load(open(f'{base}/data/processed/gtex_reference.json'))

# Compute dysregulation on external data
dysreg_genes_available = [g for g in gtex_ref.keys() if g in expr_log2_ext.columns]
dysreg_ext = pd.DataFrame(index=expr_log2_ext.index)
for gene in dysreg_genes_available:
    gtex_mean = gtex_ref[gene]['mean']
    gtex_std  = gtex_ref[gene]['std']
    if gtex_std > 0:
        dysreg_ext[gene] = (expr_log2_ext[gene] - gtex_mean) / gtex_std
    else:
        dysreg_ext[gene] = 0.0

# ── RANK dysregulation scores ─────────────────────────────────────
dysreg_ranked_ext = dysreg_ext.rank(axis=1, pct=True)
dysreg_ranked_ext = dysreg_ranked_ext.loc[common_ext]

print(f"External dysreg ranked: {dysreg_ranked_ext.shape}")

# Load immune features computed in NB15
# Reuse immune_ext from previous notebook — reload LM22 and rerun
lm22 = pd.read_csv(f'{base}/data/external/LM22.txt', sep='\t', index_col=0)
common_lm22  = lm22.index.intersection(expr_log2_ext.columns)
expr_lm22_ext = expr_log2_ext.loc[common_ext][common_lm22]
lm22_common  = lm22.loc[common_lm22]

print(f"Running CIBERSORT on {len(expr_lm22_ext)} patients...")
print(f"Start: {datetime.now().strftime('%H:%M:%S')}")

def run_cibersort_single(patient_expr, lm22_matrix):
    expr_linear = (2 ** patient_expr.values) - 1
    expr_linear = np.clip(expr_linear, 0, 1e6)
    lm22_linear = (2 ** lm22_matrix.values) - 1
    lm22_linear = np.clip(lm22_linear, 0, 1e6)
    lm22_norm   = normalize(lm22_linear, axis=0)
    expr_norm   = normalize(expr_linear.reshape(1, -1))[0]
    best_nu = 0.5; best_error = np.inf
    for nu in [0.25, 0.5, 0.75]:
        try:
            svr = NuSVR(nu=nu, kernel='linear', C=1.0)
            svr.fit(lm22_norm, expr_norm)
            error = np.mean((lm22_norm @ svr.coef_[0] - expr_norm) ** 2)
            if error < best_error:
                best_error = error; best_nu = nu
        except: continue
    try:
        svr = NuSVR(nu=best_nu, kernel='linear', C=1.0)
        svr.fit(lm22_norm, expr_norm)
        raw_weights = svr.coef_[0]
    except:
        raw_weights = np.zeros(lm22_matrix.shape[1])
    clipped = np.maximum(raw_weights, 0)
    if clipped.sum() == 0:
        clipped, _ = nnls(lm22_norm, expr_norm)
        clipped = np.maximum(clipped, 0)
    total = clipped.sum()
    final = clipped / total if total > 0 else np.ones(len(clipped)) / len(clipped)
    return dict(zip(lm22_matrix.columns, final))

results_ext = {}
for i, pid in enumerate(expr_lm22_ext.index):
    results_ext[pid] = run_cibersort_single(expr_lm22_ext.loc[pid], lm22_common)
    if (i+1) % 100 == 0 or i == 0:
        print(f"  {i+1}/{len(expr_lm22_ext)} [{datetime.now().strftime('%H:%M:%S')}]")

immune_ext_df = pd.DataFrame(results_ext).T
print(f"Immune features: {immune_ext_df.shape} ✅")

# Build ranked external feature matrix
expr_ext_lasso = pd.DataFrame(index=common_ext)
for g in lasso_genes:
    if g in expr_ranked_ext.columns:
        expr_ext_lasso[f"{g}_expr"] = expr_ranked_ext.loc[common_ext, g].values
    else:
        expr_ext_lasso[f"{g}_expr"] = 0.5  # median rank for missing genes

# Dysregulation — use ranked
# First get top_dysreg from training
cox_pvals_d_train = {}
for gene in dysreg_ranked.columns:
    try:
        df_tmp = pd.DataFrame({'T': y['time'], 'E': y['event'],
                               'gene': dysreg_ranked[gene].values})
        cph = CoxPHFitter()
        cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
        cox_pvals_d_train[gene] = cph.summary['p'].values[0]
    except:
        cox_pvals_d_train[gene] = 1.0
top_dysreg_ranked = list(pd.Series(cox_pvals_d_train).nsmallest(20).index)

dysreg_ext_sel = pd.DataFrame(index=common_ext)
for g in top_dysreg_ranked:
    if g in dysreg_ranked_ext.columns:
        dysreg_ext_sel[f"{g}_dysreg"] = dysreg_ranked_ext[g].values
    else:
        dysreg_ext_sel[f"{g}_dysreg"] = 0.5

# Interaction features
inter_ext = pd.DataFrame({
    'stageIII_x_M2':   (clinical_ext_df['stage_Stage III'] *
                        immune_ext_df['Macrophages M2']).values,
    'stageIV_x_CD8':   (clinical_ext_df['stage_Stage IV'] *
                        immune_ext_df['T cells CD8']).values,
    'age_x_stageIII':  (clinical_ext_df['age'] *
                        clinical_ext_df['stage_Stage III']).values,
    'stageIII_x_Treg': (clinical_ext_df['stage_Stage III'] *
                        immune_ext_df['T cells regulatory (Tregs)']).values,
    'M2_x_CD8':        (immune_ext_df['Macrophages M2'] *
                        immune_ext_df['T cells CD8']).values,
}, index=common_ext)

X_ext_ranked = pd.concat([expr_ext_lasso, dysreg_ext_sel,
                            immune_ext_df, clinical_ext_df,
                            inter_ext], axis=1).fillna(0.5)

# Build training ranked matrix
expr_tr_ranked = expr_ranked[lasso_genes].copy()
expr_tr_ranked.columns = [f"{g}_expr" for g in lasso_genes]

dysreg_tr_ranked = dysreg_ranked[top_dysreg_ranked].copy()
dysreg_tr_ranked.columns = [f"{g}_dysreg" for g in top_dysreg_ranked]

immune_train_full = immune.loc[common]
clinical_tr_full  = clinical_features.loc[common]

inter_tr = pd.DataFrame({
    'stageIII_x_M2':   (clinical_tr_full['stage_Stage III'] *
                        immune_train_full['Macrophages M2']).values,
    'stageIV_x_CD8':   (clinical_tr_full['stage_Stage IV'] *
                        immune_train_full['T cells CD8']).values,
    'age_x_stageIII':  (clinical_tr_full['age'] *
                        clinical_tr_full['stage_Stage III']).values,
    'stageIII_x_Treg': (clinical_tr_full['stage_Stage III'] *
                        immune_train_full['T cells regulatory (Tregs)']).values,
    'M2_x_CD8':        (immune_train_full['Macrophages M2'] *
                        immune_train_full['T cells CD8']).values,
}, index=common)

X_train_ranked = pd.concat([expr_tr_ranked, dysreg_tr_ranked,
                              immune_train_full, clinical_tr_full,
                              inter_tr], axis=1).fillna(0)

# Reorder external to match training
X_ext_ranked = X_ext_ranked[X_train_ranked.columns]

print(f"\nTraining ranked matrix: {X_train_ranked.shape}")
print(f"External ranked matrix: {X_ext_ranked.shape}")
print(f"Columns match: {list(X_ext_ranked.columns) == list(X_train_ranked.columns)}")

# Scale and predict
scaler      = StandardScaler()
X_train_s   = pd.DataFrame(scaler.fit_transform(X_train_ranked),
                             columns=X_train_ranked.columns)
X_ext_s     = pd.DataFrame(scaler.transform(X_ext_ranked),
                             columns=X_ext_ranked.columns)

model_ranked = GradientBoostingSurvivalAnalysis(
    n_estimators=300, learning_rate=0.05, max_depth=2,
    min_samples_split=20, min_samples_leaf=10,
    subsample=0.8, random_state=42)
model_ranked.fit(X_train_s, y)

risk_ext = model_ranked.predict(X_ext_s)

y_ext_arr = np.array(
    [(vs == 'Dead', float(t)) for vs, t in
     zip(survival_df_ext['vital_status'],
         survival_df_ext['survival_days'])],
    dtype=[('event', bool), ('time', float)])

ci_ext_ranked = concordance_index_censored(
    y_ext_arr['event'].astype(bool),
    y_ext_arr['time'],
    risk_ext)[0]

print(f"\n{'='*50}")
print(f"GSE68465 EXTERNAL VALIDATION — RANKED FEATURES")
print(f"{'='*50}")
print(f"C-index (ranked features):   {ci_ext_ranked:.3f}")
print(f"C-index (original features): 0.637")
print(f"Improvement:                 {ci_ext_ranked - 0.637:+.3f}")
print(f"{'='*50}")

External dysreg ranked: (442, 497)
Running CIBERSORT on 442 patients...
Start: 18:08:20
  1/442 [18:08:21]
  100/442 [18:08:24]
  200/442 [18:08:27]
  300/442 [18:08:31]
  400/442 [18:08:34]
Immune features: (442, 22) ✅

Training ranked matrix: (478, 124)
External ranked matrix: (442, 124)
Columns match: True

GSE68465 EXTERNAL VALIDATION — RANKED FEATURES
C-index (ranked features):   0.623
C-index (original features): 0.637
Improvement:                 -0.014
